# Cardiology Multilabel Classification

This notebook shows how to build a multilabel ECG classification pipeline with the PhysioNet 2020 cardiology dataset in PyHealth. It loads the dataset, applies the `CardiologyMultilabelClassification` task, inspects example samples, and includes a small Member 2 sanity check that verifies a model can run a forward pass and backpropagate on a tiny 5-sample subset.

Download link: https://physionet.org/content/challenge-2020/1.0.2/
You'll need to run the following in terminal:

`cd ~/data && wget -r -N -c -np https://physionet.org/files/challenge-2020/1.0.2/`

## 1. Install Deps

In [ ]:
%pip install -q -e ..
%load_ext autoreload
%autoreload 2

## 2. Load the Dataset

In [ ]:
from pathlib import Path

DATA_ROOT = str(Path.home() / "data" / "physionet.org" / "files" / "challenge-2020" / "1.0.2" / "training")
print(f"DATA_ROOT = {DATA_ROOT}")

In [ ]:
# Optional: clear the cache to force a full rebuild
# If you decide to change any of the core code this will be necessary to pick up changes
import shutil, os
cache_dir = "/tmp/pyhealth_cardiology"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cleared cache: {cache_dir}")
else:
    print(f"No cache found at {cache_dir}")

In [ ]:
from pyhealth.datasets import Cardiology2Dataset

dataset = Cardiology2Dataset(
    root=DATA_ROOT,
    chosen_dataset=[1, 1, 0, 0, 0, 0], # Only load cpsc_2018 datasets
    dev=True,
    cache_dir="/tmp/pyhealth_cardiology"
)

dataset.stats()

## 3. Apply the Multilabel Classification Task

In [ ]:
from pyhealth.tasks import CardiologyMultilabelClassification

task = CardiologyMultilabelClassification()
sample_dataset = dataset.set_task(task)

print(f"Total samples: {len(sample_dataset)}")

## 4. Inspect Sample

In [ ]:
sample = sample_dataset[0]
print(f"keys:       {list(sample.keys())}")
print(f"patient_id: {sample['patient_id']}")
print(f"visit_id:   {sample['visit_id']}")
print(f"signal:     {sample['signal']}")
print(f"labels:     {sample['labels']}")

## 5. Member 2 Sanity Check

In [ ]:
# %% [markdown]
# # Cardiology Multilabel Classification with ResNet-18
#
# This notebook extends the cardiology multilabel example to:
# 1. load the PhysioNet 2020 cardiology dataset
# 2. apply the Cardiology multilabel task
# 3. split the data into train/val/test sets
# 4. convert ECG windows into pseudo-images for torchvision ResNet-18
# 5. train and evaluate a ResNet-18 baseline
# 6. run a simple lead ablation experiment

# %% [markdown]
# ## 1. Install Deps

# %%
%pip install -q -e ..
%load_ext autoreload
%autoreload 2

# %% [markdown]
# ## 2. Imports and Config

# %%
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch

from pyhealth.datasets import (
    Cardiology2Dataset,
    create_sample_dataset,
    get_dataloader,
    split_by_patient,
)
from pyhealth.models import TorchvisionModel
from pyhealth.tasks import CardiologyMultilabelClassification
from pyhealth.trainer import Trainer

SEED = 42
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3
TRAIN_RATIOS = [0.7, 0.1, 0.2]
USE_PRETRAINED = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Full data yields ~2000+ batches/epoch at batch 32; each ResNet step on CPU is slow, so tqdm can
# stay at 0% until the first batch finishes. Cap steps for interactive runs; None = full epoch.
STEPS_PER_EPOCH = 500

torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_ROOT = str(
    Path.home() / "data" / "physionet.org" / "files" / "challenge-2020" / "1.0.2" / "training"
)
CACHE_DIR = "/tmp/pyhealth_cardiology"

print("DATA_ROOT =", DATA_ROOT)
print("DEVICE =", DEVICE)

# %% [markdown]
# ## 3. Load the Dataset
#
# Set `DEV = False` for the real experiment.
# Set `DEV = True` if you only want a quick smoke test.

# %%
DEV = False

dataset = Cardiology2Dataset(
    root=DATA_ROOT,
    chosen_dataset=[1, 1, 0, 0, 0, 0],  # adjust if you want more subsets
    dev=DEV,
    cache_dir=CACHE_DIR,
)

dataset.stats()

# %% [markdown]
# ## 4. Apply the Cardiology Multilabel Task

# %%
task = CardiologyMultilabelClassification()
sample_dataset = dataset.set_task(task)

print("Total windowed samples:", len(sample_dataset))
print("Number of labels:", sample_dataset.output_processors["labels"].size())

# %% [markdown]
# ## 5. Inspect One Sample

# %%
sample = sample_dataset[0]
print("keys      :", list(sample.keys()))
print("patient_id:", sample["patient_id"])
print("visit_id  :", sample["visit_id"])
print("signal shape:", np.asarray(sample["signal"]).shape)
print("labels    :", sample["labels"])

# %% [markdown]
# ## 6. Split into Train / Val / Test
#
# We split by patient to reduce leakage across windows from the same recording/patient.

# %%
train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset,
    ratios=TRAIN_RATIOS,
    seed=SEED,
)

print("Train samples:", len(train_dataset))
print("Val samples  :", len(val_dataset))
print("Test samples :", len(test_dataset))

# %% [markdown]
# ## 7. Convert ECG Windows to Pseudo-Images for torchvision
#
# Each ECG window starts as `(12, 1250)`.
# We convert it to `(1, 12, 1250)`, which lets the PyHealth torchvision wrapper
# repeat the single channel to 3 channels internally for ResNet-18.

# %%
def _multihot_to_label_list(labels, labels_processor):
    # MultiLabelProcessor expects SNOMED code strings; task samples already use tensors.
    if isinstance(labels, list):
        return labels
    idx_to_code = {i: code for code, i in labels_processor.label_vocab.items()}
    if hasattr(labels, "detach"):
        flat = labels.detach().cpu().reshape(-1)
        return [
            idx_to_code[int(j)]
            for j in range(flat.numel())
            if float(flat[j]) > 0.5
        ]
    flat = np.asarray(labels, dtype=np.float64).reshape(-1)
    return [idx_to_code[int(j)] for j in range(flat.size) if flat[j] > 0.5]


def to_resnet_ready_samples(dataset_split, labels_processor):
    converted = []
    for i in range(len(dataset_split)):
        s = copy.deepcopy(dataset_split[i])
        signal = np.asarray(s["signal"], dtype=np.float32)

        # Per-window z-score normalization
        signal = (signal - signal.mean()) / (signal.std() + 1e-8)

        # Convert from (12, 1250) to (1, 12, 1250)
        s["signal"] = np.expand_dims(signal, axis=0)
        s["labels"] = _multihot_to_label_list(s["labels"], labels_processor)
        converted.append(s)
    return converted


_labels_processor = sample_dataset.output_processors["labels"]
train_samples_img = to_resnet_ready_samples(train_dataset, _labels_processor)
val_samples_img = to_resnet_ready_samples(val_dataset, _labels_processor)
test_samples_img = to_resnet_ready_samples(test_dataset, _labels_processor)

shared_output_processors = sample_dataset.output_processors

train_img_dataset = create_sample_dataset(
    samples=train_samples_img,
    input_schema={"signal": "tensor"},
    output_schema=task.output_schema,
    dataset_name="cardiology_resnet_train",
    output_processors=shared_output_processors,
)

val_img_dataset = create_sample_dataset(
    samples=val_samples_img,
    input_schema={"signal": "tensor"},
    output_schema=task.output_schema,
    dataset_name="cardiology_resnet_val",
    output_processors=shared_output_processors,
)

test_img_dataset = create_sample_dataset(
    samples=test_samples_img,
    input_schema={"signal": "tensor"},
    output_schema=task.output_schema,
    dataset_name="cardiology_resnet_test",
    output_processors=shared_output_processors,
)

batch = next(iter(get_dataloader(train_img_dataset, batch_size=4, shuffle=False)))
print("ResNet input batch shape:", tuple(batch["signal"].shape))
print("Target batch shape      :", tuple(batch["labels"].shape))

# %% [markdown]
# ## 8. Build ResNet-18 from torchvision

# %%
weights = "DEFAULT" if USE_PRETRAINED else None

model = TorchvisionModel(
    dataset=train_img_dataset,
    model_name="resnet18",
    model_config={"weights": weights},
)

model

# %% [markdown]
# ## 9. Build Dataloaders

# %%
train_loader = get_dataloader(train_img_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = get_dataloader(val_img_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = get_dataloader(test_img_dataset, batch_size=BATCH_SIZE, shuffle=False)

_n_train_batches = len(train_loader)
if STEPS_PER_EPOCH is not None:
    print(
        f"train_loader: {_n_train_batches} batches/epoch; "
        f"steps_per_epoch={STEPS_PER_EPOCH}"
    )
else:
    print(
        f"train_loader: {_n_train_batches} batches/epoch (full pass; slow on CPU)"
    )

# %% [markdown]
# ## 10. Train ResNet-18
#
# The paper description mentions macro ROC-AUC, so we monitor that here.

# %%
metrics = [
    "roc_auc_macro",
    "pr_auc_macro",
    "f1_macro",
    "jaccard_macro",
    "hamming_loss",
]

trainer = Trainer(
    model=model,
    metrics=metrics,
    device=DEVICE,
    output_path="./output",
    exp_name="cardiology_multilabel_resnet18",
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=EPOCHS,
    optimizer_params={"lr": LR},
    steps_per_epoch=STEPS_PER_EPOCH,
    monitor="roc_auc_macro",
    monitor_criterion="max",
    load_best_model_at_last=True,
)

# %% [markdown]
# ## 11. Evaluate on Validation and Test Sets

# %%
val_results = trainer.evaluate(val_loader)
test_results = trainer.evaluate(test_loader)

print("Validation Results")
print(pd.Series(val_results).sort_index())

print("\nTest Results")
print(pd.Series(test_results).sort_index())

# %% [markdown]
# ## 12. Optional: Compare Against the Paper
#
# Fill in the paper numbers once you confirm the exact table/setting you want to reproduce.

# %%
paper_results = {
    # "roc_auc_macro": ...,
    # "pr_auc_macro": ...,
}

if paper_results:
    comparison = pd.DataFrame(
        {
            "paper": pd.Series(paper_results),
            "our_resnet18": pd.Series(test_results),
        }
    )
    comparison["delta"] = comparison["our_resnet18"] - comparison["paper"]
    display(comparison.sort_index())
else:
    print("Add the paper metrics here once you identify the exact comparison table.")

# %% [markdown]
# ## 13. Ablation: 12-Lead vs 1-Lead
#
# This reruns the same pipeline with a different `leads` argument in the task.

# %%
def run_resnet_experiment(
    leads,
    experiment_name,
    dev=DEV,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    use_pretrained=USE_PRETRAINED,
    steps_per_epoch=STEPS_PER_EPOCH,
):
    task = CardiologyMultilabelClassification(leads=leads)

    base_dataset = Cardiology2Dataset(
        root=DATA_ROOT,
        chosen_dataset=[1, 1, 0, 0, 0, 0],
        dev=dev,
        cache_dir=CACHE_DIR,
    )

    full_dataset = base_dataset.set_task(task)

    train_ds, val_ds, test_ds = split_by_patient(
        full_dataset,
        ratios=TRAIN_RATIOS,
        seed=SEED,
    )

    labels_processor = full_dataset.output_processors["labels"]

    def convert(split_ds):
        out = []
        for i in range(len(split_ds)):
            s = copy.deepcopy(split_ds[i])
            signal = np.asarray(s["signal"], dtype=np.float32)
            signal = (signal - signal.mean()) / (signal.std() + 1e-8)
            s["signal"] = np.expand_dims(signal, axis=0)
            s["labels"] = _multihot_to_label_list(s["labels"], labels_processor)
            out.append(s)
        return out

    shared_output_processors = full_dataset.output_processors

    train_img = create_sample_dataset(
        samples=convert(train_ds),
        input_schema={"signal": "tensor"},
        output_schema=task.output_schema,
        dataset_name=f"{experiment_name}_train",
        output_processors=shared_output_processors,
    )
    val_img = create_sample_dataset(
        samples=convert(val_ds),
        input_schema={"signal": "tensor"},
        output_schema=task.output_schema,
        dataset_name=f"{experiment_name}_val",
        output_processors=shared_output_processors,
    )
    test_img = create_sample_dataset(
        samples=convert(test_ds),
        input_schema={"signal": "tensor"},
        output_schema=task.output_schema,
        dataset_name=f"{experiment_name}_test",
        output_processors=shared_output_processors,
    )

    train_loader = get_dataloader(train_img, batch_size=batch_size, shuffle=True)
    val_loader = get_dataloader(val_img, batch_size=batch_size, shuffle=False)
    test_loader = get_dataloader(test_img, batch_size=batch_size, shuffle=False)

    weights = "DEFAULT" if use_pretrained else None
    model = TorchvisionModel(
        dataset=train_img,
        model_name="resnet18",
        model_config={"weights": weights},
    )

    trainer = Trainer(
        model=model,
        metrics=metrics,
        device=DEVICE,
        output_path="./output",
        exp_name=experiment_name,
    )

    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=epochs,
        optimizer_params={"lr": lr},
        steps_per_epoch=steps_per_epoch,
        monitor="roc_auc_macro",
        monitor_criterion="max",
        load_best_model_at_last=True,
    )

    return trainer.evaluate(test_loader)

# %%
ablation_results = {
    "12-lead": run_resnet_experiment(
        leads=list(range(12)),
        experiment_name="cardiology_resnet18_12lead",
    ),
    "1-lead": run_resnet_experiment(
        leads=[0],
        experiment_name="cardiology_resnet18_1lead",
    ),
}

ablation_df = pd.DataFrame(ablation_results).T
display(ablation_df)

# %% [markdown]
# ## 14. Clean Up Temporary In-Memory Datasets

# %%
for ds in [train_img_dataset, val_img_dataset, test_img_dataset]:
    ds.close()
